In [2]:
from scipy.sparse import load_npz
import numpy as np

In [4]:
X_train_tfidf = load_npz("Processed data/X_train_tfidf.npz")
X_test_tfidf = load_npz("Processed data/X_test_tfidf.npz")

y_train = np.load("Processed data/y_train.npy")
y_test = np.load("Processed data/y_test.npy")

In [5]:
print("X_train:", X_train_tfidf.shape)
print("X_test:", X_test_tfidf.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (14942, 5000)
X_test: (2675, 5000)
y_train: (14942,)
y_test: (2675,)


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [7]:
ann_model = Sequential([
    Dense(128, activation="relu", input_shape=(5000,)),
    Dropout(0.3),

    Dense(64, activation="relu"),
    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

c:\Users\Yashi\tf_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
ann_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       640,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,449 (2.47 MB)

 Trainable params: 648,449 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
train_idx, val_idx = train_test_split(
    range(X_train_tfidf.shape[0]),
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

In [12]:
X_train_ann = X_train_tfidf[train_idx]
X_val_ann = X_train_tfidf[val_idx]

In [13]:
y_train_ann = y_train[train_idx]
y_val_ann = y_train[val_idx]

In [14]:
print(X_train_ann.shape)
print(X_val_ann.shape)

(13447, 5000)
(1495, 5000)


In [15]:
batch_size = 32

def data_generator(X, y):
    for start in range(0, X.shape[0], batch_size):
        X_batch = X[start:start + batch_size].toarray().astype("float32")
        y_batch = y[start:start + batch_size].astype("float32")

        yield X_batch, y_batch

In [21]:
train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(X_train_ann, y_train_ann),
    output_signature=(
        tf.TensorSpec(shape=(None, 5000), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32)
    )
).repeat()

In [22]:
val_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(X_val_ann, y_val_ann),
    output_signature=(
        tf.TensorSpec(shape=(None, 5000), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32)
    )
).repeat()

In [19]:
steps_per_epoch = int(np.ceil(X_train_ann.shape[0] / batch_size))
validation_steps = int(np.ceil(X_val_ann.shape[0] / batch_size))

print("Training steps:", steps_per_epoch)
print("Validation steps:", validation_steps)

Training steps: 421
Validation steps: 47


In [23]:
ann_model = Sequential([
    Dense(128, activation="relu", input_shape=(5000,)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

c:\Users\Yashi\tf_env\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:
ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [26]:
history = ann_model.fit(
    train_dataset,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_dataset,
    validation_steps=validation_steps,
    epochs=10,
    verbose=1
)

Epoch 1/10
421/421 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7131 - loss: 0.5796 - val_accuracy: 0.7472 - val_loss: 0.5370
Epoch 2/10
421/421 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7577 - loss: 0.5031 - val_accuracy: 0.7271 - val_loss: 0.5454
Epoch 3/10
421/421 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.7831 - loss: 0.4592 - val_accuracy: 0.7204 - val_loss: 0.5623
Epoch 4/10
421/421 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8046 - loss: 0.4249 - val_accuracy: 0.7258 - val_loss: 0.5803
Epoch 5/10
421/421 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8163 - loss: 0.3939 - val_accuracy: 0.7177 - val_loss: 0.6192
Epoch 6/10
421/421 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8321 - loss: 0.3644 - val_accuracy: 0.7117 - val_loss: 0.6566
Epoch 7/10
421/421 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8432 - loss: 0.3396 - val_accuracy: 0.7157 - val_loss: 0.6915
Epoch 8/10
421/421 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8451 - loss: 0.3214 - val_accuracy: 0

In [28]:
def test_generator(X, y):
    for start in range(0, X.shape[0], batch_size):
        X_batch = X[start:start + batch_size].toarray().astype("float32")
        y_batch = y[start:start + batch_size].astype("float32")

        yield X_batch, y_batch

In [29]:
test_dataset = tf.data.Dataset.from_generator(
    lambda: test_generator(X_test_tfidf, y_test),
    output_signature=(
        tf.TensorSpec(shape=(None, 5000), dtype=tf.float32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32)
    )
)

In [31]:
test_steps = int(np.ceil(X_test_tfidf.shape[0] / batch_size))

print("Test samples:", X_test_tfidf.shape[0])
print("Test steps:", test_steps)

Test samples: 2675
Test steps: 84


In [30]:
y_pred_prob = ann_model.predict(
    test_dataset,
    steps=int(np.ceil(X_test_tfidf.shape[0] / batch_size)),
    verbose=1
)

y_pred_ann = (y_pred_prob.ravel() >= 0.5).astype(int)

print("Predictions:", y_pred_ann.shape)

84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Predictions: (2675,)


In [32]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("ANN Accuracy:", accuracy_score(y_test, y_pred_ann))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ann))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ann))

ANN Accuracy: 0.7057943925233645

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.71      0.76      1732
           1       0.57      0.69      0.62       943

    accuracy                           0.71      2675
   macro avg       0.69      0.70      0.69      2675
weighted avg       0.72      0.71      0.71      2675


Confusion Matrix:
[[1238  494]
 [ 293  650]]


In [33]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "SVM",
        "Random Forest",
        "ANN"
    ],
    "Accuracy": [
        0.7686,
        0.7439,
        0.7779,
        0.7809,
        0.7058
    ],
    "Hallucination Precision": [
        0.70,
        0.64,
        0.74,
        0.74,
        0.57
    ],
    "Hallucination Recall": [
        0.60,
        0.61,
        0.57,
        0.59,
        0.69
    ],
    "Hallucination F1": [
        0.65,
        0.63,
        0.64,
        0.65,
        0.62
    ]
})

results

,Model,Accuracy,Hallucination Precision,Hallucination Recall,Hallucination F1
0,Logistic Regression,0.7686,0.70,0.60,0.65
1,Naive Bayes,0.7439,0.64,0.61,0.63
2,SVM,0.7779,0.74,0.57,0.64
3,Random Forest,0.7809,0.74,0.59,0.65
4,ANN,0.7058,0.57,0.69,0.62
